In [ ]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split 
from sklearn.model_selection import cross_val_score 
from sklearn.metrics import accuracy_score 
from sklearn.compose import ColumnTransformer 
from sklearn.pipeline import Pipeline 
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import OneHotEncoder 


In [ ]:
import os

os.listdir('/kaggle/input')

## Load Dataset


In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/test.csv")

print(train_df.shape)
print(test_df.shape)

## Preview Dataset


In [ ]:
train_df.head()
train_df.info()

## Missing Values

In [ ]:
train_df.isnull().sum()

## Feature Engineering
### Split Passenger Id

In [ ]:
train_df[['Group', 'GroupNumber']] = train_df['PassengerId'].str.split('_', expand=True) 
test_df[['Group', 'GroupNumber']] = test_df['PassengerId'].str.split('_', expand=True)

### Split Cabin

In [ ]:
train_df[['Deck', 'CabinNum', 'Side']] = train_df['Cabin'].str.split('/', expand=True) 
test_df[['Deck', 'CabinNum', 'Side']] = test_df['Cabin'].str.split('/', expand=True)

## Drop Unnecessary Columns

In [ ]:
train_df = train_df.drop(columns=['Name', 'Cabin'])

test_df = test_df.drop(columns=['Name', 'Cabin'])

## Separate Features and Target

In [ ]:
X = train_df.drop("Transported", axis=1) 
y = train_df["Transported"]

## Identify Numerical and Categorical Columns

In [ ]:
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns 

categorical_cols = X.select_dtypes(include=['object', 'bool']).columns 

print("Numerical Columns:")
print(numerical_cols) 

print("\nCategorical Columns:")
print(categorical_cols)

## Preprocessing Pipelines

### Numerical Pipelines

In [ ]:
numerical_transformer = Pipeline(steps=[ ('imputer', SimpleImputer(strategy='median')) ])

### Categorical Pipeline

In [ ]:
categorical_transformer = Pipeline(steps=[ ('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore')) ])

## Combine Preprocessing

In [ ]:
preprocessor = ColumnTransformer( transformers=[ ('num', numerical_transformer, numerical_cols), ('cat', categorical_transformer, categorical_cols) ] )

# Build Model Pipeline


In [ ]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.02,
    depth=8,
    loss_function='Logloss',
    eval_metric='Accuracy',
    random_seed=42,
    verbose=200
)

## Create Full Pipeline

In [ ]:
clf = Pipeline(steps=[ ('preprocessor', preprocessor), ('model', model) ])

## Train Validation Split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split( X, y, test_size=0.2, random_state=42 )

## Train Model 

In [ ]:

clf.fit(X_train, y_train)

## Validation Predictions

In [ ]:
preds = clf.predict(X_valid)

## Accuracy Score

In [ ]:
acc = accuracy_score(y_valid, preds) 
print("Validation Accuracy:", acc)

#### Random forest = 74.58%
#### XGB Boost = 78.89 %
#### CatBoost = 79.12 %


## Train on Full Data

In [ ]:
clf.fit(X, y)

## Predict Test Data

In [ ]:
test_preds = clf.predict(test_df)

## Submission File

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Transported": test_preds
})

submission.to_csv("submission.csv", index=False)

print("submission.csv created successfully!")

In [ ]:
submission.head()